# Primer — *What is a graph?*

The visual for the **"What is a graph"** slide. No LLM, no extraction, no network — we just
**hand-build a tiny predefined graph** and draw it, so the picture matches the four words on the
slide: **nodes, edges, weights, properties.**

It's deliberately clean and idealised — the *opposite* of the messy extracted graph on the next
slide. Concept first, reality (the yolo graph) second.

Built with the same node/edge conventions as the rest of `graphtools` (`kind` / `label` on nodes,
`rel` on edges), so it doubles as a reusable **fixture** for the Vue/SVG renderer later.

## Build the predefined graph

A tiny recipe fragment: one **recipe**, three **ingredients**, one **step** — wired by *directed,
typed* edges. Each primitive the slide names is present in the data:

- **nodes** — the things (recipe / ingredient / step), each with a `kind` and a `label`;
- **edges** — the directed relationships between them (`CONTAINS`, `HAS_STEP`);
- **weights** — a number on an edge (here the quantity, e.g. flour `weight=200`);
- **properties** — arbitrary attributes on nodes/edges (e.g. flour's `category`, the edge `unit`).

In [ ]:
import networkx as nx


def primer_graph() -> nx.MultiDiGraph:
    """A tiny, hand-built recipe graph for the 'what is a graph' primer."""
    g = nx.MultiDiGraph()

    # NODES — each has a `kind` (drives colour) and a `label`; some carry extra PROPERTIES.
    g.add_node("recipe:pancakes", kind="recipe", label="Pancakes", serves=4)
    g.add_node("ingredient:flour", kind="ingredient", label="Flour", category="dry good")
    g.add_node("ingredient:milk", kind="ingredient", label="Milk", category="dairy")
    g.add_node("ingredient:egg", kind="ingredient", label="Egg", category="dairy")
    g.add_node("step:whisk", kind="step", label="Whisk together")

    # EDGES — directed + typed (`rel`); CONTAINS carries a WEIGHT (quantity) + a PROPERTY (unit).
    g.add_edge("recipe:pancakes", "ingredient:flour", rel="CONTAINS", weight=200, unit="g")
    g.add_edge("recipe:pancakes", "ingredient:milk", rel="CONTAINS", weight=300, unit="ml")
    g.add_edge("recipe:pancakes", "ingredient:egg", rel="CONTAINS", weight=2, unit="")
    g.add_edge("recipe:pancakes", "step:whisk", rel="HAS_STEP", order=1)
    return g


g = primer_graph()
print(f"{g.number_of_nodes()} nodes, {g.number_of_edges()} edges\n")

# The data behind the picture — nodes, edges, weights, properties, made concrete.
print("NODES (with properties):")
for n, d in g.nodes(data=True):
    props = {k: v for k, v in d.items() if k not in ('kind', 'label')}
    print(f"  {d['label']:<16} kind={d['kind']:<11} {props}")

print("\nEDGES (with weights + properties):")
for u, v, d in g.edges(data=True):
    props = ', '.join(f'{k}={v!r}' for k, v in d.items() if k != 'rel')
    print(f"  {u} -[{d['rel']}]-> {v}   ({props})")

## Draw it

Render with the shared `graphtools.viz.render_graph` (deterministic matplotlib layout) so the
primer matches the look of every other graph in the talk. Saves `primer_graph.png` next to this
notebook so you can see the shape.

> *Now you try:* add a node (say `ingredient:sugar`) and a `CONTAINS` edge to it, re-run, and watch
> it join the picture.

In [ ]:
from IPython.display import Image
from graphtools.viz import render_graph

render_graph(g, path="primer_graph.png")
Image(filename="primer_graph.png")